In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_applicants = 1000

data = {
    'credit_score': np.random.randint(500, 850, n_applicants),
    'annual_income': np.random.randint(30000, 150000, n_applicants),
    'demographic_group': np.random.choice([0, 1], n_applicants, p=[0.4, 0.6]) # 0 = Group A, 1 = Group B
}

df = pd.DataFrame(data)

# Inject biased historical labels: Group 0 gets approved less often for the same credit score
df['approved'] = np.where(
    (df['credit_score'] > 650) & (df['annual_income'] > 50000) & (df['demographic_group'] == 1), 1,
    np.where((df['credit_score'] > 720) & (df['annual_income'] > 80000), 1, 0)
)
print(df.head())

   credit_score  annual_income  demographic_group  approved
0           602          71832                  1         0
1           848          88596                  1         1
2           770          60523                  0         0
3           606         118861                  1         0
4           571         116740                  1         0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = df[['credit_score', 'annual_income', 'demographic_group']]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

# Append model predictions back to the evaluation set
test_results = X_test.copy()
test_results['true_approved'] = y_test
test_results['predicted_approved'] = model.predict(X_test)

In [3]:
# Calculate selection rate for Group 0
group_0 = test_results[test_results['demographic_group'] == 0]
rate_0 = group_0['predicted_approved'].mean()

# Calculate selection rate for Group 1
group_1 = test_results[test_results['demographic_group'] == 1]
rate_1 = group_1['predicted_approved'].mean()

disparate_impact = rate_0 / rate_1
print(f"Group 0 Approval Rate: {rate_0:.2%}")
print(f"Group 1 Approval Rate: {rate_1:.2%}")
print(f"Disparate Impact Ratio: {disparate_impact:.2f}")

if disparate_impact < 0.8:
    print("Warning: Algorithmic Bias detected against Group 0!")
else:
    print("Model passes basic statistical fairness thresholds.")

Group 0 Approval Rate: 41.56%
Group 1 Approval Rate: 49.59%
Disparate Impact Ratio: 0.84
Model passes basic statistical fairness thresholds.
